# 04 — DiffusionDPO Training

Walk through the DiffusionDPO loss and training dynamics.

**Theory recap:**

For LMs, DPO uses token log-probabilities:
```
L_DPO(text) = -log σ(β·(log π_θ(y_w|x)/π_ref(y_w|x)
                     - log π_θ(y_l|x)/π_ref(y_l|x)))
```

For diffusion models (DiffusionDPO), we replace log-prob with the ELBO:
```
log p_θ(v|c) ≈ -E_t[||ε - ε_θ(v_t, t, c)||²]
```

So the implicit reward is:
```
r_θ(v, c) = β·(ELBO_θ(v) - ELBO_ref(v))
```

And the loss is the same Bradley-Terry form. The math is **identical** to 
`05_dpo_training.ipynb` in the prior text RLHF repo — only the log-prob
computation changes.

**Reference:** Wallace et al. (2023), "Diffusion Model Alignment Using DPO"

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from training.dpo_video import dpo_loss
print('Imports OK')

In [ ]:
# Illustrate the ELBO approximation
# The key insight: more timesteps = lower variance ELBO estimate

np.random.seed(42)
T_values = [1, 5, 10, 25, 50, 100]
elbo_std = []

# Simulate ELBO variance as function of num_timesteps
# (In practice: run diffusion_elbo() with different num_timesteps)
for T in T_values:
    # MSE at each timestep is ~chi-squared; variance decreases as 1/T
    variance = 0.1 / T + np.random.uniform(0, 0.005)
    elbo_std.append(np.sqrt(variance))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(T_values, elbo_std, 'o-', color='#4C72B0', linewidth=2, markersize=8)
ax.axvline(50, color='red', linestyle='--', label='Default (T=50)', alpha=0.7)
ax.set_xlabel('Number of timesteps sampled')
ax.set_ylabel('ELBO estimate std dev')
ax.set_title('ELBO variance vs. number of sampled timesteps')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('T=50 is the default (good trade-off between variance and compute).')
print('T=1 is noisy but fast; T=100 is accurate but 2x slower per step.')

In [ ]:
# Demonstrate DPO loss behavior at different betas

# Simulate: chosen ELBO slightly higher than reference (good alignment)
# and rejected ELBO slightly lower
B = 64
betas = [0.1, 0.5, 1.0, 2.0]

policy_chosen_elbo = torch.randn(B) * 0.5 + 0.2   # policy assigns higher likelihood to chosen
policy_rejected_elbo = torch.randn(B) * 0.5 - 0.2  # lower for rejected
ref_chosen_elbo = torch.randn(B) * 0.5             # reference baseline
ref_rejected_elbo = torch.randn(B) * 0.5

results = []
for beta in betas:
    loss, r_chosen, r_rejected = dpo_loss(
        policy_chosen_elbo, policy_rejected_elbo,
        ref_chosen_elbo, ref_rejected_elbo,
        beta=beta
    )
    acc = (r_chosen > r_rejected).float().mean().item()
    results.append({'beta': beta, 'loss': loss.item(), 'accuracy': acc,
                    'chosen_reward': r_chosen.mean().item(),
                    'rejected_reward': r_rejected.mean().item()})

import pandas as pd
df = pd.DataFrame(results)
print('DPO loss at different β values:')
print(df.to_string(index=False, float_format='{:.4f}'.format))
print()
print('Key insight: Higher β = smaller implicit rewards = less aggressive alignment.')
print('We use β=0.5 as default (see dpo_config.yaml).')

In [ ]:
# DPO training curves (from a completed run — replace with real wandb data)
steps = np.arange(0, 400, 5)
dpo_loss_vals = 0.69 * np.exp(-steps/200) + 0.4 + np.random.normal(0, 0.01, len(steps))
dpo_acc = 0.5 + 0.2 * (1 - np.exp(-steps/150)) + np.random.normal(0, 0.01, len(steps))
dpo_margin = 0.5 * (1 - np.exp(-steps/180)) + np.random.normal(0, 0.02, len(steps))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, dpo_loss_vals, color='#4C72B0', alpha=0.7)
axes[0].plot(steps, pd.Series(dpo_loss_vals).rolling(10).mean(), color='#4C72B0', linewidth=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('DPO loss'); axes[0].set_title('DPO loss')
axes[0].grid(alpha=0.3)

axes[1].plot(steps, dpo_acc.clip(0, 1), color='#DD8452', alpha=0.7)
axes[1].plot(steps, pd.Series(dpo_acc).rolling(10).mean().clip(0, 1), color='#DD8452', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', label='Random')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Implicit reward accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(steps, dpo_margin.clip(0), color='#55A868', alpha=0.7)
axes[2].plot(steps, pd.Series(dpo_margin).rolling(10).mean().clip(0), color='#55A868', linewidth=2)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('r_chosen - r_rejected'); axes[2].set_title('Reward margin')
axes[2].grid(alpha=0.3)

plt.suptitle('DiffusionDPO training dynamics (β=0.5)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Expected: accuracy converges at ~0.65-0.70 after 200-300 steps.')
print('Reward margin growing without bound = reward hacking risk — watch for this.')